# Advanced Keras Deep Learning Constructs
This notebook demonstrates advanced Keras concepts requested in the assignment.

Topics:
1. Custom Learning Rate Scheduler
2. Custom Dropout
3. Custom Normalization
4. TensorBoard
5. Custom Loss Function
6. Custom Activation, Initializer, Regularizer, Constraint
7. Custom Metric
8. Custom Layers
9. Custom Model
10. Custom Optimizer
11. Custom Training Loop


In [ ]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np


## Load Dataset

In [ ]:

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 28*28)
x_test = x_test.reshape(-1, 28*28)

print("Training shape:", x_train.shape)


## Custom Activation Function

In [ ]:

def leaky_relu(z):
    return tf.maximum(0.01 * z, z)


## Custom Initializer

In [ ]:

def my_glorot_initializer(shape, dtype=tf.float32):
    stddev = tf.sqrt(2. / (shape[0] + shape[1]))
    return tf.random.normal(shape, stddev=stddev, dtype=dtype)


## Custom Regularizer

In [ ]:

class MyL1Regularizer(keras.regularizers.Regularizer):
    def __init__(self, factor=0.01):
        self.factor = factor

    def __call__(self, weights):
        return self.factor * tf.reduce_sum(tf.abs(weights))


## Custom Weight Constraint

In [ ]:

class MyPositiveWeights(keras.constraints.Constraint):
    def __call__(self, weights):
        return tf.maximum(weights, 0.)


## Custom Dropout Layer

In [ ]:

class CustomDropout(layers.Layer):
    def __init__(self, rate):
        super().__init__()
        self.rate = rate

    def call(self, inputs, training=None):
        if training:
            return tf.nn.dropout(inputs, rate=self.rate)
        return inputs


## Custom Normalization Layer

In [ ]:

class MaxNormDense(layers.Dense):
    def call(self, inputs):
        self.kernel.assign(tf.clip_by_norm(self.kernel, 1.0))
        return super().call(inputs)


## Custom Loss Function (Huber)

In [ ]:

def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    is_small = tf.abs(error) < delta
    squared = 0.5 * tf.square(error)
    linear = delta * tf.abs(error) - 0.5 * delta**2
    return tf.where(is_small, squared, linear)


## Custom Metric

In [ ]:

class HuberMetric(keras.metrics.Metric):
    def __init__(self, name="huber_metric", **kwargs):
        super().__init__(name=name, **kwargs)
        self.total = self.add_weight("total", initializer="zeros")
        self.count = self.add_weight("count", initializer="zeros")

    def update_state(self, y_true, y_pred):
        loss = tf.reduce_mean(huber_loss(y_true, y_pred))
        self.total.assign_add(loss)
        self.count.assign_add(1.0)

    def result(self):
        return self.total / self.count


## Custom Layer Example

In [ ]:

class AddGaussianNoise(layers.Layer):
    def __init__(self, stddev):
        super().__init__()
        self.stddev = stddev

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(tf.shape(inputs), stddev=self.stddev)
            return inputs + noise
        return inputs


## Custom Model using subclassing

In [ ]:

class ResidualBlock(layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.dense1 = layers.Dense(units, activation="relu")
        self.dense2 = layers.Dense(units)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        return inputs + x


class ResidualModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.noise = AddGaussianNoise(0.1)
        self.block = ResidualBlock(64)
        self.out = layers.Dense(10, activation="softmax")

    def call(self, inputs):
        x = self.noise(inputs)
        x = self.block(x)
        return self.out(x)


## Custom Optimizer

In [ ]:

class MyMomentumOptimizer(keras.optimizers.SGD):
    def __init__(self, learning_rate=0.01, momentum=0.9):
        super().__init__(learning_rate=learning_rate, momentum=momentum)


## Custom Learning Rate Scheduler

In [ ]:

def scheduler(epoch, lr):
    return lr * 0.95

lr_callback = keras.callbacks.LearningRateScheduler(scheduler)


## TensorBoard Callback

In [ ]:

tensorboard_cb = keras.callbacks.TensorBoard(log_dir="logs")


## Train Model

In [ ]:

model = ResidualModel()

model.compile(
    optimizer=MyMomentumOptimizer(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.2,
    callbacks=[lr_callback, tensorboard_cb]
)


## Custom Training Loop

In [ ]:

optimizer = keras.optimizers.Adam()
loss_fn = keras.losses.SparseCategoricalCrossentropy()

dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

for epoch in range(2):
    print("Epoch:", epoch)
    for step, (x_batch, y_batch) in enumerate(dataset):

        with tf.GradientTape() as tape:
            logits = model(x_batch)
            loss = loss_fn(y_batch, logits)

        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        if step % 200 == 0:
            print("step", step, "loss", float(loss))
